In [ ]:
from pyspark.sql import SparkSession


# col (selecionar coluna), concat_ws (juntar listas), to_utc_timestamp (arrumar datas)

from pyspark.sql.functions import col, concat_ws, to_utc_timestamp

#iniciou sessão

spark = SparkSession.builder \
  .appName("ETL_Univ_Pyspark") \
  .getOrCreate()

dados_brutos_academico = [
      {
        "aluno_nome": "Gabriel",
        "matricula": 1001,
        # 'curso' é um dicionário escondido dentro do JSON principal
        "curso": {"nome": "Engenharia", "modalidade": "Presencial", "campus": "Curitiba"},
        # 'disciplinas' é um dicionário que contém uma lista dentro dele
        "disciplinas": {"cursando": ["Sistemas Distribuídos", "Banco de Dados"], "ead": "N", "bolsista": True},
        # 'acessos' contém a lista de catracas por onde o aluno passou hoje
        "acessos": {"catracas": ["Portaria Principal", "Biblioteca Central"]},
        # Data vem como um texto comum com fuso horário local (-03:00)
        "data_matricula": "2026-02-10T08:00:00.000-03:00"
      },
      {
        "aluno_nome": "João Silva",
        "matricula": 1002,
        "curso": {"nome": "Sistemas de Informação", "modalidade": "EAD", "campus": "Online"},
        "disciplinas": {"cursando": ["Arquitetura de Software", "Engenharia de Dados"], "ead": "S", "bolsista": False},
        "acessos": {"catracas": []},
        "data_matricula": "2026-02-12T14:30:00.000-03:00"
      }
  ]



In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, ArrayType

# Definição explícita do esquema (schema) para evitar erros de inferência automática
schema = StructType([
    StructField("aluno_nome", StringType(), True),
    StructField("matricula", IntegerType(), True),
    StructField("curso", StructType([
        StructField("nome", StringType(), True),
        StructField("modalidade", StringType(), True),
        StructField("campus", StringType(), True)
    ]), True),
    StructField("disciplinas", StructType([
        StructField("cursando", ArrayType(StringType()), True),
        StructField("ead", StringType(), True),
        StructField("bolsista", BooleanType(), True)
    ]), True),
    StructField("acessos", StructType([
        StructField("catracas", ArrayType(StringType()), True)
    ]), True),
    StructField("data_matricula", StringType(), True)
])

# Criando o DataFrame usando o esquema explícito
df_bruto = spark.createDataFrame(dados_brutos_academico, schema=schema)

#Transform
df_transformado = df_bruto.select(
    col("aluno_nome"),
    col("matricula"),
    col("curso.nome").alias("curso_nome"),
    col("curso.modalidade").alias("curso_modalidade"),
    col("curso.campus").alias("curso_campus"),
    concat_ws(", ", col("disciplinas.cursando")).alias("disciplinas_cursando"),
    col("disciplinas.ead").alias("flag_ead"),
    col("disciplinas.bolsista").alias("flag_bolsista"),
    concat_ws(", ", col("acessos.catracas")).alias("historico_catracas"),
    to_utc_timestamp(col("data_matricula"), "UTC").alias("data_matricula_utc")
)

#imprimi o resultado
print("Dados Transformados (Limpos e Prontos para o Banco de Dados):")
df_transformado.show(truncate=False)

#Load
#Salva a tabela limpa no formato Parquet
df_transformado.write.mode("overwrite").parquet("dados_alunos_pucpr_parquet")

print("Pipeline ETL finalizado com sucesso!")

Dados Transformados (Limpos e Prontos para o Banco de Dados):
+----------+---------+----------------------+----------------+------------+--------------------------------------------+--------+-------------+--------------------------------------+-------------------+
|aluno_nome|matricula|curso_nome            |curso_modalidade|curso_campus|disciplinas_cursando                        |flag_ead|flag_bolsista|historico_catracas                    |data_matricula_utc |
+----------+---------+----------------------+----------------+------------+--------------------------------------------+--------+-------------+--------------------------------------+-------------------+
|Gabriel   |1001     |Engenharia            |Presencial      |Curitiba    |Sistemas Distribuídos, Banco de Dados       |N       |true         |Portaria Principal, Biblioteca Central|2026-02-10 11:00:00|
|João Silva|1002     |Sistemas de Informação|EAD             |Online      |Arquitetura de Software, Engenharia de Dados|S     